In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Try to import RDKit
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
    from rdkit.ML.Descriptors import MoleculeDescriptors
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False
    print("Error: RDKit not installed. Please install RDKit to run predictions.")
    exit(1)

class CBM_Predictor_Fixed:
    def __init__(self, model_path):

        self.model_path = model_path
        self.loaded_model = None
        self.scaler = None
        self.feature_dimension = None
        self.feature_names = []
        self.selected_feature_indices = None
        self.dim_reducer = None
        self.dim_reduction_method = None
        
        self.load_model_and_features()
    
    def load_model_and_features(self):
        model_file = os.path.join(self.model_path, 'best_advanced_model.pkl')
        
        if not os.path.exists(model_file):
            raise FileNotFoundError(f"Model file not found: {model_file}")
        
        print(f"Loading model from: {model_file}")
        model_info = joblib.load(model_file)
        
        self.loaded_model = model_info['model']
        self.scaler = model_info['scaler']
        self.best_params = model_info.get('best_params', {})
        self.best_model_name = model_info.get('best_model_name', 'Unknown')
        self.feature_dimension = model_info.get('feature_dimension', None)
        self.test_r2 = model_info.get('test_r2', 0.0)
        self.test_rmse = model_info.get('test_rmse', 0.0)
        
        self.dim_reducer = model_info.get('dim_reducer', None)
        self.selected_feature_indices = model_info.get('selected_feature_indices', None)
        self.original_feature_names = model_info.get('original_feature_names', [])
        
        print(f"✓ Model loaded: {self.best_model_name}")
        print(f"✓ Model R² on test set: {self.test_r2:.4f}")
        print(f"✓ Feature dimension: {self.feature_dimension}")
        
        if self.dim_reducer is not None:
            self.dim_reduction_method = type(self.dim_reducer).__name__
            print(f"✓ Loaded dimensionality reducer: {self.dim_reduction_method}")
            if hasattr(self.dim_reducer, 'n_components'):
                print(f"  Number of components: {self.dim_reducer.n_components}")
        elif self.selected_feature_indices is not None:
            self.dim_reduction_method = 'Feature Selection'
            print(f"✓ Loaded feature selection indices: {len(self.selected_feature_indices)} indices")
        else:
            self.dim_reduction_method = None
            print("ℹ No dimensionality reduction or feature selection loaded")
        
        self.load_feature_importance_info_for_reference()
    
    def load_feature_importance_info_for_reference(self):
        feature_importance_file = os.path.join(self.model_path, 'feature_importance.csv')
        
        if os.path.exists(feature_importance_file):
            print(f"Feature importance file found (for reference only): {feature_importance_file}")
            try:
                feature_importance_df = pd.read_csv(feature_importance_file, encoding='utf-8-sig')
                print(f"  File shape: {feature_importance_df.shape}")
            except Exception as e:
                print(f"  Error reading feature importance file: {e}")
    
    def advanced_feature_engineering(self, smiles_list):

        if not RDKIT_AVAILABLE:
            raise ImportError("RDKit not available, cannot perform advanced feature engineering")
        
        print("Performing advanced feature engineering...")
        
        features = []
        valid_indices = []
        
        feature_names_local = []
        
        for radius in [1, 2, 3]:
            nbits = 256 if radius < 3 else 128
            for i in range(nbits):
                feature_names_local.append(f"Morgan_R{radius}_Bit_{i}")
        
        descriptor_names = [x[0] for x in Descriptors._descList]
        feature_names_local.extend(descriptor_names)
        
        additional_descriptors = [
            'MolWt', 'HeavyAtomCount', 'NumRotatableBonds', 'NumHeteroatoms'
        ]
        feature_names_local.extend(additional_descriptors)
        
        print(f"Total original features to extract: {len(feature_names_local)}")
        
        for i, smiles in enumerate(smiles_list):
            try:
                mol = Chem.MolFromSmiles(str(smiles))
                if mol is None:
                    print(f"Warning: Could not parse SMILES: {smiles[:50]}...")
                    continue
                
                fp_1 = AllChem.GetMorganFingerprintAsBitVect(mol, 1, nBits=256)
                fp_2 = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=256)
                fp_3 = AllChem.GetMorganFingerprintAsBitVect(mol, 3, nBits=128)
                
                descriptor_calc = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)
                descriptors = descriptor_calc.CalcDescriptors(mol)
                
                mol_3d = Chem.AddHs(mol)
                try:
                    AllChem.EmbedMolecule(mol_3d)
                    AllChem.UFFOptimizeMolecule(mol_3d)
                    descriptors = list(descriptors) + [
                        Descriptors.MolWt(mol),
                        Descriptors.HeavyAtomCount(mol),
                        rdMolDescriptors.CalcNumRotatableBonds(mol),
                        rdMolDescriptors.CalcNumHeteroatoms(mol)
                    ]
                except:
                    descriptors = list(descriptors)
                
                combined_features = np.concatenate([
                    np.array(fp_1),
                    np.array(fp_2), 
                    np.array(fp_3),
                    np.array(descriptors)
                ])
                
                combined_features = np.nan_to_num(combined_features, nan=0.0, posinf=0.0, neginf=0.0)
                
                features.append(combined_features)
                valid_indices.append(i)
                
            except Exception as e:
                print(f"Error processing SMILES {smiles[:50]}...: {e}")
                continue
        
        if not features:
            raise ValueError("No valid SMILES could be processed")
        
        features_array = np.array(features)
        print(f"Successfully extracted features for {len(features)} molecules")
        print(f"Original feature dimensions: {features_array.shape}")
        
        if len(features_array) >= 2:
            print("\nOriginal Feature Discrepancy Check:")
            for i in range(min(3, len(features_array))):
                print(f"  molecule {i}: Feature mean={np.mean(features_array[i]):.6f}, "
                      f"Feature standard deviation={np.std(features_array[i]):.6f}")
            
            if len(features_array) >= 2:
                diff_01 = np.sum(np.abs(features_array[0] - features_array[1]))
                print(f"  molecule0 vs molecule1 Sum of absolute differences: {diff_01:.6f}")
                
                if len(features_array) >= 3:
                    diff_02 = np.sum(np.abs(features_array[0] - features_array[2]))
                    diff_12 = np.sum(np.abs(features_array[1] - features_array[2]))
                    print(f"  molecule0 vs molecule2 Sum of absolute differences: {diff_02:.6f}")
                    print(f"  molecule1 vs molecule2 Sum of absolute differences: {diff_12:.6f}")
        
        return features_array, valid_indices
        
    
    def reduce_features_to_30d(self, features):
        print(f"Reducing features from {features.shape[1]}D to {self.feature_dimension}D...")
    
        if self.dim_reducer is not None:
            print(f"Attempting to use saved {type(self.dim_reducer).__name__}...")
            if hasattr(self.dim_reducer, 'transform'):
                try:
                    reduced = self.dim_reducer.transform(features)
                    print(f"✓ Successfully used dim_reducer.transform()")
                    print(f"  Reduced shape: {reduced.shape}")
                    print(f"  Reduced feature stats: mean={reduced.mean():.6f}, std={reduced.std():.6f}")
                    return reduced
                except Exception as e:
                    print(f"✗ Error using dim_reducer.transform: {e}")
            else:
                print(f"✗ Dim reducer lacks transform method")
    
        if self.selected_feature_indices is not None:
            print(f"Attempting to use saved feature selection indices (len={len(self.selected_feature_indices)})...")
            max_idx = features.shape[1] - 1
            valid_indices = [idx for idx in self.selected_feature_indices if 0 <= idx <= max_idx]
            print(f"  Valid indices count: {len(valid_indices)} (out of {len(self.selected_feature_indices)})")
            if len(valid_indices) == self.feature_dimension:
                reduced = features[:, valid_indices]
                print(f"✓ Successfully used feature selection indices")
                print(f"  Reduced shape: {reduced.shape}")
                print(f"  Reduced feature stats: mean={reduced.mean():.6f}, std={reduced.std():.6f}")
                return reduced
            else:
                print(f"✗ Index count mismatch or out of range. Expected {self.feature_dimension}, got {len(valid_indices)} valid.")
    
        print("WARNING: No valid dim_reducer or feature indices found! Using variance-based fallback.")
        feature_variances = np.var(features, axis=0)
        selected_indices = np.argsort(feature_variances)[-self.feature_dimension:]
        selected_indices = np.sort(selected_indices)
        reduced = features[:, selected_indices]
        print(f"Fallback selected {reduced.shape[1]} features")
        print(f"Fallback reduced stats: mean={reduced.mean():.6f}, std={reduced.std():.6f}")
        return reduced
    
    def extract_and_reduce_features(self, smiles_list):

        if not RDKIT_AVAILABLE:
            raise ImportError("RDKit not available, cannot perform feature extraction")
        
        print(f"Extracting features for {len(smiles_list)} molecules...")
        
        features, valid_indices = self.advanced_feature_engineering(smiles_list)
        
        print(f"\nOriginal features dimension: {features.shape[1]}")
        print(f"Model expected dimension: {self.feature_dimension}")
        
        if features.shape[1] != self.feature_dimension:
            features = self.reduce_features_to_30d(features)
        else:
            print("✓ Feature dimensions already match")
        
        return features, valid_indices
    
    def predict_cbm(self, smiles_list):

        print(f"\nPredicting CBM for {len(smiles_list)} molecules...")

        features, valid_indices = self.extract_and_reduce_features(smiles_list)
        
        if features.shape[0] == 0:
            return [], []

        if features.shape[1] != self.feature_dimension:
            print(f"✗ Error: Features dimension ({features.shape[1]}) doesn't match model dimension ({self.feature_dimension})")
            return [], valid_indices
        else:
            print(f"✓ Feature dimension matches model expectation")
        
        if self.scaler:
            print("\nApplying feature scaling...")
            features_before_scaling = features.copy()
            features = self.scaler.transform(features)
            

            if features.shape[0] >= 2:
                print("Before and after scaling comparison:")
                for i in range(min(3, features.shape[0])):
                    mean_before = np.mean(features_before_scaling[i])
                    mean_after = np.mean(features[i])
                    print(f"  molecule {i}: Mean before scaling={mean_before:.6f}, Mean after scaling={mean_after:.6f}, "
                          f"Zoom level={mean_after/mean_before if mean_before != 0 else 'N/A':.6f}")
        
        print("\nMaking predictions with model...")
        predictions = self.loaded_model.predict(features)
        
        print(f"\nForecast results:")
        for i in range(min(3, len(predictions))):
            print(f"  molecule {i}: {predictions[i]:.6f}")
        
        unique_predictions = np.unique(np.round(predictions, 6))
        if len(unique_predictions) == 1:
            print(f"⚠️ Warning: All predicted values are the same! Value = {unique_predictions[0]}")
        else:
            print(f"✓ Different predicted values, normal variation")
            print(f"  Predicted value range: [{np.min(predictions):.6f}, {np.max(predictions):.6f}]")
        
        print(f"Predictions completed for {len(predictions)} molecules")
        
        return predictions, valid_indices
    
    def predict_from_csv(self, csv_path, output_path=None, smiles_column=None):

        print(f"\nLoading data from: {csv_path}")
        
        if csv_path.endswith('.csv'):
            df = pd.read_csv(csv_path)
        else:
            df = pd.read_excel(csv_path)
        
        print(f"Dataset shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        
        if smiles_column is None:
            columns = df.columns.tolist()
            smiles_col = next((col for col in columns if 'smile' in col.lower()), None)
            if smiles_col is None:
                smiles_col = next((col for col in columns if 'SMILES' in col), None)
            if smiles_col is None and len(columns) > 0:
                smiles_col = columns[0]
        else:
            smiles_col = smiles_column
        
        if smiles_col not in df.columns:
            raise ValueError(f"SMILES column '{smiles_col}' not found in dataset")
        
        print(f"Using SMILES column: {smiles_col}")
        
        smiles_list = df[smiles_col].astype(str).tolist()
        print(f"SMILES Examples (First 3): {smiles_list[:3] if len(smiles_list) >= 3 else smiles_list}")
        
        original_columns = df.columns.tolist()
        
        predictions, valid_indices = self.predict_cbm(smiles_list)
        
        if len(predictions) == 0:
            print("No predictions were made")
            return pd.DataFrame()

        results_data = []
        for i, idx in enumerate(valid_indices):
            result_row = {}

            for col in original_columns:
                result_row[col] = df.iloc[idx][col]
            
            result_row['CBM_Predicted'] = predictions[i]
            result_row['Prediction_Index'] = i
            
            results_data.append(result_row)
        
        predictions_df = pd.DataFrame(results_data)
        
        col_order = original_columns + ['Prediction_Index', 'CBM_Predicted']
        predictions_df = predictions_df[col_order]
        
        if output_path:
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            predictions_df.to_csv(output_path, index=False, encoding='utf-8-sig')
            print(f"\nPredictions saved to: {output_path}")
        
        return predictions_df


if __name__ == "__main__":
    model_path = r"E:\a\dou\SAM\vs3\cbm"
    input_file = r"E:\a\dou\SAM\vs3\verify\verify.csv"
    output_file = r"E:\a\dou\SAM\vs3\predictions\cbm_predictions_fixed.csv"
    
    print("=" * 80)
    print("CBM Prediction System - Fixed Version (Use the saved dimensionality reducer)")
    print("=" * 80)

    if not os.path.exists(model_path):
        print(f"Error: Model path does not exist: {model_path}")
        exit(1)
    
    if not os.path.exists(input_file):
        print(f"Error: Input file does not exist: {input_file}")
        exit(1)
    
    output_dir = os.path.dirname(output_file)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    try:
        predictor = CBM_Predictor_Fixed(model_path)
        
        print(f"\nProcessing input file: {input_file}")
        results = predictor.predict_from_csv(
            csv_path=input_file,
            output_path=output_file,
            smiles_column=None
        )
        
        if len(results) > 0:
            print("\n" + "=" * 80)
            print("Prediction Results Summary:")
            print("=" * 80)
            print(f"Total molecules in input: {len(pd.read_csv(input_file))}")
            print(f"Successfully predicted: {len(results)}")
            
            print(f"\nPrediction Statistics:")
            print(f"Mean predicted CBM: {results['CBM_Predicted'].mean():.6f}")
            print(f"Std predicted CBM: {results['CBM_Predicted'].std():.6f}")
            print(f"Min predicted CBM: {results['CBM_Predicted'].min():.6f}")
            print(f"Max predicted CBM: {results['CBM_Predicted'].max():.6f}")
            
            print(f"\nAll predictions:")
            for i, row in results.iterrows():
                mol_name = row.get('name', f"Molecule_{i+1}")
                smiles_short = str(row.get('smiles', 'N/A'))[:50]
                if len(str(row.get('smiles', ''))) > 50:
                    smiles_short += "..."
                
                print(f"  {i+1}. {mol_name}")
                print(f"     SMILES: {smiles_short}")
                print(f"     Predicted CBM: {row['CBM_Predicted']:.6f}")
        
        print(f"\nDetailed results saved to: {output_file}")
        
    except Exception as e:
        print(f"\nError during prediction: {str(e)}")
        import traceback
        traceback.print_exc()
        exit(1)
    
    print("\n" + "=" * 80)
    print("Prediction completed!")
    print("=" * 80)

CBM Prediction System - Fixed Version (使用保存的降维器)
Loading model from: E:\a\dou\SAM\vs3\cbm\best_advanced_model.pkl
✓ Model loaded: VotingRegressor
✓ Model R² on test set: 0.9126
✓ Feature dimension: 30
✓ Loaded dimensionality reducer: PLSRegression
  Number of components: 30
Feature importance file found (for reference only): E:\a\dou\SAM\vs3\cbm\feature_importance.csv
  File shape: (30, 2)

Processing input file: E:\a\dou\SAM\vs3\verify\verify.csv

Loading data from: E:\a\dou\SAM\vs3\verify\verify.csv
Dataset shape: (6, 2)
Columns: ['name', 'smiles']
Using SMILES column: smiles
SMILES示例 (前3个): ['COc1ccc(N(c2ccc(OC)cc2)c2ccc(-c3ccc(-c4ccc(C(=O)O)cc4)c4nsnc34)cc2)cc1', 'COc1ccc(N(c2ccc(OC)cc2)c2ccc(-c3ccc(-c4cc(C(=O)O)cc(C(=O)O)c4)c4nsnc34)cc2)cc1', 'COc1ccc(N(c2ccc(OC)cc2)c2ccc(-c3ccc(-c4ccc(C(=O)O)c(F)c4)c4nsnc34)cc2)cc1']

Predicting CBM for 6 molecules...
Extracting features for 6 molecules...
Performing advanced feature engineering...
Total original features to extract: 861


[23:00:10] DEPRECATION WARNING: please use MorganGenerator
[23:00:10] DEPRECATION WARNING: please use MorganGenerator
[23:00:10] DEPRECATION WARNING: please use MorganGenerator
[23:00:11] DEPRECATION WARNING: please use MorganGenerator
[23:00:11] DEPRECATION WARNING: please use MorganGenerator
[23:00:11] DEPRECATION WARNING: please use MorganGenerator
[23:00:11] DEPRECATION WARNING: please use MorganGenerator
[23:00:11] DEPRECATION WARNING: please use MorganGenerator
[23:00:11] DEPRECATION WARNING: please use MorganGenerator
[23:00:11] DEPRECATION WARNING: please use MorganGenerator
[23:00:11] DEPRECATION WARNING: please use MorganGenerator
[23:00:11] DEPRECATION WARNING: please use MorganGenerator
[23:00:11] DEPRECATION WARNING: please use MorganGenerator
[23:00:11] DEPRECATION WARNING: please use MorganGenerator
[23:00:11] DEPRECATION WARNING: please use MorganGenerator
[23:00:11] DEPRECATION WARNING: please use MorganGenerator
[23:00:11] DEPRECATION WARNING: please use MorganGenerat

Successfully extracted features for 6 molecules
Original feature dimensions: (6, 861)

原始特征差异检查:
  分子 0: 特征均值=4411839.241971, 特征标准差=129380309.856225
  分子 1: 特征均值=15020091.327578, 特征标准差=440475311.305848
  分子 2: 特征均值=6263571.444139, 特征标准差=183683750.824260
  分子0 vs 分子1 绝对差总和: 9133705242.169165
  分子0 vs 分子2 绝对差总和: 1594341614.368755
  分子1 vs 分子2 绝对差总和: 7539363759.643673

Original features dimension: 861
Model expected dimension: 30
Reducing features from 861D to 30D...
Attempting to use saved PLSRegression...
✓ Successfully used dim_reducer.transform()
  Reduced shape: (6, 30)
  Reduced feature stats: mean=33.276134, std=230.144024
✓ Feature dimension matches model expectation

Making predictions with model...

预测结果:
  分子 0: -0.738892
  分子 1: 3.088632
  分子 2: -0.091238
✓ 预测值不同，差异正常
  预测值范围: [-2.962787, 3.088632]
Predictions completed for 6 molecules

Predictions saved to: E:\a\dou\SAM\vs3\predictions\cbm_predictions_fixed.csv

Prediction Results Summary:
Total molecules in input: 6
Successf

In [ ]:
import joblib

model_path = r"E:\a\dou\SAM\vs3\cbm\best_advanced_model.pkl"
info = joblib.load(model_path)
print("Keys in model_info:", info.keys())
print("dim_reducer present:", info.get('dim_reducer') is not None)
print("selected_feature_indices present:", info.get('selected_feature_indices') is not None)
print("original_feature_names length:", len(info.get('original_feature_names', [])))
print("feature_dimension (model expects):", info.get('feature_dimension'))

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Try to import RDKit
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
    from rdkit.ML.Descriptors import MoleculeDescriptors
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False
    print("Error: RDKit not installed. Please install RDKit to run predictions.")
    exit(1)

class VBM_Predictor_Fixed:
    def __init__(self, model_path):
        """
        初始化VBM预测器
        
        Args:
            model_path: 保存的模型路径，包含best_advanced_model.pkl文件
        """
        self.model_path = model_path
        self.loaded_model = None
        self.scaler = None
        self.feature_dimension = None
        self.feature_names = []
        self.selected_feature_indices = None
        self.dim_reducer = None
        self.dim_reduction_method = None
        
        # 加载模型和特征处理信息
        self.load_model_and_features()
    
    def load_model_and_features(self):
        """加载保存的模型和特征处理信息"""
        # 检查是文件还是目录
        if os.path.isfile(self.model_path):
            model_file = self.model_path
        elif os.path.isdir(self.model_path):
            model_file = os.path.join(self.model_path, 'best_advanced_model.pkl')
        else:
            raise FileNotFoundError(f"Model path not found: {self.model_path}")
        
        if not os.path.exists(model_file):
            raise FileNotFoundError(f"Model file not found: {model_file}")
        
        print(f"Loading model from: {model_file}")
        model_info = joblib.load(model_file)
        
        self.loaded_model = model_info['model']
        self.scaler = model_info['scaler']
        self.best_params = model_info.get('best_params', {})
        self.best_model_name = model_info.get('best_model_name', 'Unknown')
        self.feature_dimension = model_info.get('feature_dimension', None)
        self.test_r2 = model_info.get('test_r2', 0.0)
        self.test_rmse = model_info.get('test_rmse', 0.0)
        
        print(f"Model loaded: {self.best_model_name}")
        print(f"Model R² on test set: {self.test_r2:.4f}")
        print(f"Model RMSE on test set: {self.test_rmse:.4f}")
        print(f"Feature dimension: {self.feature_dimension}")
        
        # 尝试从报告文件中获取降维方法信息
        self.load_dimension_reduction_info()
        
        # 尝试从特征重要性文件中获取特征信息
        self.load_feature_importance_info()
    
    def load_dimension_reduction_info(self):
        """从报告文件中获取降维方法信息"""
        # 确定报告文件路径
        if os.path.isfile(self.model_path):
            model_dir = os.path.dirname(self.model_path)
        else:
            model_dir = self.model_path
        
        report_file = os.path.join(model_dir, 'comprehensive_analysis_report.txt')
        
        if os.path.exists(report_file):
            try:
                with open(report_file, 'r', encoding='utf-8') as f:
                    content = f.read()
                    
                    # 查找降维方法
                    if "Dimensionality Reduction Method: PCA" in content:
                        self.dim_reduction_method = 'PCA'
                        print("Detected PCA dimensionality reduction from report")
                    elif "Dimensionality Reduction Method: PLS" in content:
                        self.dim_reduction_method = 'PLS'
                        print("Detected PLS dimensionality reduction from report")
                    elif "Feature Selection" in content:
                        self.dim_reduction_method = 'Feature Selection'
                        print("Detected Feature Selection from report")
                    else:
                        # 尝试从内容中提取
                        lines = content.split('\n')
                        for line in lines:
                            if "Dimensionality Reduction Method:" in line:
                                method = line.split(":")[-1].strip()
                                self.dim_reduction_method = method
                                print(f"Detected {method} from report")
                                break
                        
            except Exception as e:
                print(f"Error reading report file: {e}")
    
    def load_feature_importance_info(self):
        """加载特征重要性信息"""
        # 确定特征重要性文件路径
        if os.path.isfile(self.model_path):
            model_dir = os.path.dirname(self.model_path)
        else:
            model_dir = self.model_path
        
        feature_importance_file = os.path.join(model_dir, 'feature_importance.csv')
        
        if os.path.exists(feature_importance_file):
            print(f"Loading feature importance from: {feature_importance_file}")
            try:
                feature_importance_df = pd.read_csv(feature_importance_file, encoding='utf-8-sig')
                
                # 检查文件结构
                print(f"Feature importance file columns: {feature_importance_df.columns.tolist()}")
                print(f"Feature importance file shape: {feature_importance_df.shape}")
                
                # 根据不同的文件格式处理
                if 'Feature_Index' in feature_importance_df.columns:
                    # 这是降维后的特征
                    print("Feature importance file contains Feature_Index (likely after dimensionality reduction)")
                    
                    # 获取特征索引
                    self.selected_feature_indices = feature_importance_df['Feature_Index'].values
                    print(f"Loaded {len(self.selected_feature_indices)} feature indices")
                    
                elif 'Feature_Name' in feature_importance_df.columns:
                    # 这是特征选择后的特征
                    print("Feature importance file contains Feature_Name (likely after feature selection)")
                    
                    # 获取特征名称
                    self.feature_names = feature_importance_df['Feature_Name'].tolist()
                    print(f"Loaded {len(self.feature_names)} feature names")
                    
                    # 获取前几个特征名称示例
                    if len(self.feature_names) > 0:
                        print(f"Sample feature names: {self.feature_names[:5]}")
                
            except Exception as e:
                print(f"Error loading feature importance file: {e}")
        else:
            print("Feature importance file not found")
    
    def advanced_feature_engineering(self, smiles_list):
        """
        高级特征工程（与训练代码相同）
        """
        if not RDKIT_AVAILABLE:
            raise ImportError("RDKit not available, cannot perform advanced feature engineering")
        
        print("Performing advanced feature engineering...")
        
        features = []
        valid_indices = []
        processed_smiles = []
        
        # 生成特征名称（与训练代码相同）
        self.original_feature_names = []
        
        # 指纹特征名称
        for radius in [1, 2, 3]:
            nbits = 256 if radius < 3 else 128
            for i in range(nbits):
                self.original_feature_names.append(f"Morgan_R{radius}_Bit_{i}")
        
        # 分子描述符名称
        descriptor_names = [x[0] for x in Descriptors._descList]
        self.original_feature_names.extend(descriptor_names)
        
        # 额外的3D描述符名称
        additional_descriptors = [
            'MolWt', 'HeavyAtomCount', 'NumRotatableBonds', 'NumHeteroatoms'
        ]
        self.original_feature_names.extend(additional_descriptors)
        
        print(f"Total original features to extract: {len(self.original_feature_names)}")
        
        for i, smiles in enumerate(smiles_list):
            try:
                mol = Chem.MolFromSmiles(str(smiles))
                if mol is None:
                    print(f"Warning: Could not parse SMILES: {smiles[:50]}...")
                    # 添加零向量作为占位符
                    zero_vector = np.zeros(len(self.original_feature_names))
                    features.append(zero_vector)
                    valid_indices.append(i)
                    processed_smiles.append(smiles)
                    continue
                
                # 1. 多重指纹组合
                fp_1 = AllChem.GetMorganFingerprintAsBitVect(mol, 1, nBits=256)
                fp_2 = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=256)
                fp_3 = AllChem.GetMorganFingerprintAsBitVect(mol, 3, nBits=128)
                
                # 2. 扩展分子描述符
                descriptor_calc = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)
                descriptors = descriptor_calc.CalcDescriptors(mol)
                
                # 3. 3D描述符（估算）
                mol_3d = Chem.AddHs(mol)
                try:
                    AllChem.EmbedMolecule(mol_3d)
                    AllChem.UFFOptimizeMolecule(mol_3d)
                    # 添加一些与3D相关的特征估算
                    descriptors = list(descriptors) + [
                        Descriptors.MolWt(mol),
                        Descriptors.HeavyAtomCount(mol),
                        rdMolDescriptors.CalcNumRotatableBonds(mol),
                        rdMolDescriptors.CalcNumHeteroatoms(mol)
                    ]
                except:
                    descriptors = list(descriptors)
                
                # 4. 合并所有特征
                combined_features = np.concatenate([
                    np.array(fp_1),
                    np.array(fp_2), 
                    np.array(fp_3),
                    np.array(descriptors)
                ])
                
                # 处理可能的NaN值
                combined_features = np.nan_to_num(combined_features, nan=0.0, posinf=0.0, neginf=0.0)
                
                features.append(combined_features)
                valid_indices.append(i)
                processed_smiles.append(smiles)
                
            except Exception as e:
                print(f"Error processing SMILES {smiles[:50]}...: {e}")
                # 添加零向量作为占位符
                zero_vector = np.zeros(len(self.original_feature_names))
                features.append(zero_vector)
                valid_indices.append(i)
                processed_smiles.append(smiles)
                continue
        
        if not features:
            raise ValueError("No valid SMILES could be processed")
        
        features_array = np.array(features)
        print(f"Successfully extracted features for {len(features)} molecules")
        print(f"Original feature dimensions: {features_array.shape}")
        
        return features_array, valid_indices, processed_smiles
    
    def reduce_features(self, features):
        """
        将特征降到模型期望的维度
        """
        original_dim = features.shape[1]
        target_dim = self.feature_dimension
        
        print(f"Reducing features from {original_dim}D to {target_dim}D...")
        
        # 方法1：如果从特征重要性文件加载了特征索引，使用它们
        if self.selected_feature_indices is not None and len(self.selected_feature_indices) == target_dim:
            print("Using pre-selected feature indices...")
            
            # 确保索引在范围内
            valid_indices = [idx for idx in self.selected_feature_indices 
                           if idx < features.shape[1]]
            
            if len(valid_indices) == target_dim:
                reduced_features = features[:, valid_indices]
                print(f"Successfully selected features using pre-defined indices")
                return reduced_features
        
        # 方法2：如果从特征重要性文件加载了特征名称，使用它们
        elif self.feature_names and len(self.feature_names) == target_dim:
            print("Using pre-selected feature names...")
            
            # 我们需要找到特征名称对应的索引
            selected_indices = []
            for feature_name in self.feature_names:
                if feature_name in self.original_feature_names:
                    idx = self.original_feature_names.index(feature_name)
                    selected_indices.append(idx)
            
            if len(selected_indices) == target_dim:
                reduced_features = features[:, selected_indices]
                print(f"Successfully selected features using pre-defined names")
                return reduced_features
        
        # 方法3：如果知道是PCA降维，可以使用PCA
        elif self.dim_reduction_method == 'PCA':
            print("Using PCA for dimensionality reduction...")
            from sklearn.decomposition import PCA
            
            # 使用与训练时相同的参数
            pca = PCA(n_components=target_dim, random_state=42)
            reduced_features = pca.fit_transform(features)
            
            print(f"PCA explained variance ratio: {pca.explained_variance_ratio_.sum():.4f}")
            return reduced_features
        
        # 方法4：使用简单的方法 - 选择方差最大的特征
        else:
            print("Using variance-based feature selection...")
            
            # 计算每个特征的方差
            feature_variances = np.var(features, axis=0)
            
            # 选择方差最大的特征
            selected_indices = np.argsort(feature_variances)[-target_dim:]
            selected_indices = np.sort(selected_indices)
            
            print(f"Selected feature indices (variance-based): {selected_indices[:10]}...")
            
            reduced_features = features[:, selected_indices]
            return reduced_features
    
    def extract_and_reduce_features(self, smiles_list):
        """
        提取特征并降维到模型期望的维度
        """
        if not RDKIT_AVAILABLE:
            raise ImportError("RDKit not available, cannot perform feature extraction")
        
        print(f"Extracting features for {len(smiles_list)} molecules...")
        
        # 提取原始特征
        features, valid_indices, processed_smiles = self.advanced_feature_engineering(smiles_list)
        
        # 检查特征维度
        print(f"Original features dimension: {features.shape[1]}")
        print(f"Model expected dimension: {self.feature_dimension}")
        
        # 如果维度不匹配，进行降维
        if features.shape[1] != self.feature_dimension:
            features = self.reduce_features(features)
        
        return features, valid_indices, processed_smiles
    
    def predict_vbm(self, smiles_list):
        """
        预测VBM值
        """
        print(f"Predicting VBM for {len(smiles_list)} molecules...")
        
        # 提取并处理特征
        features, valid_indices, processed_smiles = self.extract_and_reduce_features(smiles_list)
        
        if features.shape[0] == 0:
            return [], [], []
        
        # 检查特征维度是否匹配
        if features.shape[1] != self.feature_dimension:
            print(f"Error: Features dimension ({features.shape[1]}) doesn't match model dimension ({self.feature_dimension})")
            return [], valid_indices, processed_smiles
        
        # 应用缩放（如果模型训练时使用了缩放）
        if self.scaler:
            print("Applying feature scaling...")
            features = self.scaler.transform(features)
        
        # 进行预测
        predictions = self.loaded_model.predict(features)
        
        print(f"Predictions completed for {len(predictions)} molecules")
        
        return predictions, valid_indices, processed_smiles
    
    def predict_from_file(self, input_file, output_path=None, smiles_column=None):
        """
        从文件读取SMILES并进行预测
        """
        print(f"Loading data from: {input_file}")
        
        # 读取数据
        if input_file.endswith('.csv'):
            df = pd.read_csv(input_file)
        else:
            df = pd.read_excel(input_file)
        
        print(f"Dataset shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        
        # 自动检测SMILES列
        if smiles_column is None:
            columns = df.columns.tolist()
            smiles_col = next((col for col in columns if 'smile' in col.lower()), None)
            if smiles_col is None:
                smiles_col = next((col for col in columns if 'SMILES' in col), None)
            if smiles_col is None and len(columns) > 0:
                smiles_col = columns[0]
        else:
            smiles_col = smiles_column
        
        if smiles_col not in df.columns:
            raise ValueError(f"SMILES column '{smiles_col}' not found in dataset")
        
        print(f"Using SMILES column: {smiles_col}")
        
        # 提取SMILES列表
        smiles_list = df[smiles_col].astype(str).tolist()
        
        # 获取所有列名用于结果保存
        original_columns = df.columns.tolist()
        
        # 进行预测
        predictions, valid_indices, processed_smiles = self.predict_vbm(smiles_list)
        
        if len(predictions) == 0:
            print("No predictions were made")
            return pd.DataFrame()
        
        # 创建结果DataFrame
        results_data = []
        for i, idx in enumerate(valid_indices):
            result_row = {}
            
            # 添加原始数据
            for col in original_columns:
                result_row[col] = df.iloc[idx][col]
            
            # 添加预测结果
            result_row['VBM_Predicted'] = predictions[i]
            result_row['Prediction_Index'] = i
            
            results_data.append(result_row)
        
        predictions_df = pd.DataFrame(results_data)
        
        # 重新排列列，使预测结果在最后
        col_order = original_columns + ['Prediction_Index', 'VBM_Predicted']
        predictions_df = predictions_df[col_order]
        
        # 保存结果
        if output_path:
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            predictions_df.to_csv(output_path, index=False, encoding='utf-8-sig')
            print(f"Predictions saved to: {output_path}")
        
        return predictions_df, df  # 返回预测结果和原始DataFrame


# 主程序
if __name__ == "__main__":
    # 设置路径
    model_path = r"E:\a\dou\SAM\vs3\vbm"  # 模型目录
    input_file = r"E:\a\dou\SAM\vs3\verify\verify.csv"  # 输入文件
    output_file = r"E:\a\dou\SAM\vs3\predictions\vbm_predictions_fixed.csv"  # 输出文件
    
    print("=" * 80)
    print("VBM Prediction System - Fixed Version")
    print("=" * 80)
    
    # 检查文件是否存在
    if not os.path.exists(model_path):
        print(f"Error: Model path does not exist: {model_path}")
        exit(1)
    
    if not os.path.exists(input_file):
        print(f"Error: Input file does not exist: {input_file}")
        exit(1)
    
    # 创建输出目录
    output_dir = os.path.dirname(output_file)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # 创建预测器并进行预测
    try:
        # 初始化预测器
        predictor = VBM_Predictor_Fixed(model_path)
        
        # 从文件进行预测
        print(f"\nProcessing input file: {input_file}")
        results, input_df = predictor.predict_from_file(
            input_file=input_file,
            output_path=output_file,
            smiles_column=None
        )
        
        if len(results) > 0:
            # 显示统计信息
            print("\n" + "=" * 80)
            print("Prediction Results Summary:")
            print("=" * 80)
            print(f"Total molecules in input: {len(input_df)}")
            print(f"Successfully predicted: {len(results)}")
            
            print(f"\nPrediction Statistics:")
            print(f"Mean predicted VBM: {results['VBM_Predicted'].mean():.4f}")
            print(f"Std predicted VBM: {results['VBM_Predicted'].std():.4f}")
            print(f"Min predicted VBM: {results['VBM_Predicted'].min():.4f}")
            print(f"Max predicted VBM: {results['VBM_Predicted'].max():.4f}")
            
            # 显示所有预测结果
            print(f"\nAll predictions:")
            for i, row in results.iterrows():
                mol_name = row.get('name', f"Molecule_{i+1}")
                smiles_short = str(row.get('smiles', 'N/A'))[:50]
                if len(str(row.get('smiles', ''))) > 50:
                    smiles_short += "..."
                
                print(f"  {i+1}. {mol_name}")
                print(f"     SMILES: {smiles_short}")
                print(f"     Predicted VBM: {row['VBM_Predicted']:.4f}")
                print()
        
        print(f"\nDetailed results saved to: {output_file}")
        
    except Exception as e:
        print(f"\nError during prediction: {str(e)}")
        import traceback
        traceback.print_exc()
        exit(1)
    
    print("\n" + "=" * 80)
    print("Prediction completed!")
    print("=" * 80)

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Try to import RDKit
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
    from rdkit.ML.Descriptors import MoleculeDescriptors
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False
    print("Error: RDKit not installed. Please install RDKit to run predictions.")
    exit(1)

class BindEnergy_Predictor:
    def __init__(self, model_path):
        """
        初始化BindEnergy预测器
        
        Args:
            model_path: 保存的模型路径，包含best_advanced_model.pkl文件
        """
        self.model_path = model_path
        self.loaded_model = None
        self.scaler = None
        self.pls_reducer = None
        self.feature_dimension = None
        self.descriptor_names = None
        self.feature_processors = None
        self.best_model_name = None
        
        # 加载模型和特征处理信息
        self.load_model_and_features()
    
    def load_model_and_features(self):
        """加载保存的模型和特征处理信息"""
        # 检查是文件还是目录
        if os.path.isfile(self.model_path):
            model_file = self.model_path
        elif os.path.isdir(self.model_path):
            # 尝试多个可能的模型文件名
            possible_files = [
                'best_advanced_model.pkl',
                'bindenergy_model.pkl',
                'model.pkl'
            ]
            
            for filename in possible_files:
                model_file = os.path.join(self.model_path, filename)
                if os.path.exists(model_file):
                    break
        else:
            raise FileNotFoundError(f"Model path not found: {self.model_path}")
        
        if not os.path.exists(model_file):
            raise FileNotFoundError(f"Model file not found: {model_file}")
        
        print(f"Loading model from: {model_file}")
        model_info = joblib.load(model_file)
        
        # 提取模型信息
        self.loaded_model = model_info.get('model')
        self.scaler = model_info.get('scaler')
        self.pls_reducer = model_info.get('pls_reducer')
        self.best_model_name = model_info.get('best_model_name', 'Unknown')
        self.descriptor_names = model_info.get('descriptor_names')
        self.feature_processors = model_info.get('feature_processors', {})
        
        # 获取特征维度
        if self.feature_processors:
            self.feature_dimension = self.feature_processors.get('reduced_feature_dim')
            self.original_feature_dim = self.feature_processors.get('original_feature_dim')
            self.reduction_method = self.feature_processors.get('reduction_method', 'None')
        else:
            # 如果feature_processors不存在，尝试从其他信息推断
            self.feature_dimension = model_info.get('feature_dimension')
            self.original_feature_dim = None
            self.reduction_method = 'Unknown'
        
        print(f"Model loaded: {self.best_model_name}")
        
        if self.feature_processors:
            print(f"Feature dimension (reduced): {self.feature_dimension}")
            print(f"Original feature dimension: {self.original_feature_dim}")
            print(f"Reduction method: {self.reduction_method}")
        
        # 显示模型性能信息（如果可用）
        if 'test_r2' in model_info:
            print(f"Model R² on test set: {model_info['test_r2']:.4f}")
        if 'test_rmse' in model_info:
            print(f"Model RMSE on test set: {model_info['test_rmse']:.4f}")
    
    def extract_features_from_smiles(self, smiles_list):
        """
        从SMILES字符串提取特征（与训练代码相同）
        """
        if not RDKIT_AVAILABLE:
            raise ImportError("RDKit not available, cannot perform feature extraction")
        
        print(f"Extracting features for {len(smiles_list)} molecules...")
        
        # 如果描述符名称未加载，使用默认值
        if self.descriptor_names is None:
            try:
                self.descriptor_names = [x[0] for x in Descriptors._descList]
                print(f"Using {len(self.descriptor_names)} default descriptors")
            except:
                # 如果无法获取描述符列表，使用空列表
                self.descriptor_names = []
                print("Warning: Could not load descriptor names, using empty list")
        
        features = []
        valid_indices = []
        processed_smiles = []
        
        for i, smiles in enumerate(smiles_list):
            try:
                if not isinstance(smiles, str):
                    smiles = str(smiles)
                
                mol = Chem.MolFromSmiles(smiles)
                if mol is None:
                    print(f"Warning: Could not parse SMILES: {smiles[:50]}...")
                    # 如果是第一个分子，我们需要知道特征维度
                    if i == 0 and self.original_feature_dim:
                        zero_vector = np.zeros(self.original_feature_dim)
                        features.append(zero_vector)
                        valid_indices.append(i)
                        processed_smiles.append(smiles)
                    continue
                
                # 1. 指纹（使用与训练相同的参数）
                fp_1 = AllChem.GetMorganFingerprintAsBitVect(mol, 1, nBits=256)
                fp_2 = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=256)
                fp_3 = AllChem.GetMorganFingerprintAsBitVect(mol, 3, nBits=128)
                
                # 2. 描述符
                descriptor_calc = MoleculeDescriptors.MolecularDescriptorCalculator(self.descriptor_names)
                descriptors = descriptor_calc.CalcDescriptors(mol)
                
                # 3. 添加额外的描述符（与训练代码相同）
                extra_descriptors = [
                    Descriptors.MolWt(mol),
                    Descriptors.HeavyAtomCount(mol),
                    rdMolDescriptors.CalcNumRotatableBonds(mol),
                    rdMolDescriptors.CalcNumHeteroatoms(mol),
                    Descriptors.NumHDonors(mol),
                    Descriptors.NumHAcceptors(mol),
                    Descriptors.TPSA(mol),
                    Descriptors.MolLogP(mol)
                ]
                
                # 4. 合并特征
                combined_features = np.concatenate([
                    np.array(fp_1),
                    np.array(fp_2), 
                    np.array(fp_3),
                    np.array(descriptors),
                    np.array(extra_descriptors)
                ])
                
                # 5. 检查特征维度
                if self.original_feature_dim is not None and len(combined_features) != self.original_feature_dim:
                    print(f"Warning: Feature dimension mismatch for SMILES {i}: "
                          f"expected {self.original_feature_dim}, got {len(combined_features)}")
                    # 填充或截断以匹配原始维度
                    if len(combined_features) > self.original_feature_dim:
                        combined_features = combined_features[:self.original_feature_dim]
                    else:
                        padding = np.zeros(self.original_feature_dim - len(combined_features))
                        combined_features = np.concatenate([combined_features, padding])
                
                # 6. 处理NaN值
                combined_features = np.nan_to_num(combined_features, nan=0.0)
                
                features.append(combined_features)
                valid_indices.append(i)
                processed_smiles.append(smiles)
                
            except Exception as e:
                print(f"Error processing SMILES {smiles[:50]}...: {e}")
                # 如果是第一个分子，我们需要知道特征维度
                if i == 0 and self.original_feature_dim:
                    zero_vector = np.zeros(self.original_feature_dim)
                    features.append(zero_vector)
                    valid_indices.append(i)
                    processed_smiles.append(smiles)
                continue
        
        if not features:
            raise ValueError("No valid features extracted from SMILES")
        
        features_array = np.array(features)
        print(f"Successfully extracted features for {len(features)} molecules")
        print(f"Feature matrix shape: {features_array.shape}")
        
        return features_array, valid_indices, processed_smiles
    
    def process_features(self, features):
        """
        处理特征：降维和标准化（与训练时相同）
        """
        print("Processing features...")
        
        # 1. 应用降维（如果存在）
        if self.pls_reducer is not None:
            print(f"Applying {self.reduction_method} dimensionality reduction...")
            
            # 检查降维器类型并应用相应的变换
            if hasattr(self.pls_reducer, 'transform'):
                features = self.pls_reducer.transform(features)
            elif hasattr(self.pls_reducer, 'fit_transform'):
                # 如果只有fit_transform方法，我们需要先fit再transform
                # 注意：这可能会改变降维器的状态，但不应该用于生产
                print("Warning: Using fit_transform on reducer - this may alter the reducer")
                features = self.pls_reducer.fit_transform(features)
            else:
                print("Warning: Reducer does not have transform method")
            
            print(f"Features after reduction: {features.shape}")
        else:
            print("No dimensionality reduction applied")
        
        # 2. 检查特征维度是否匹配
        if self.feature_dimension is not None and features.shape[1] != self.feature_dimension:
            print(f"Warning: Feature dimension mismatch after processing: "
                  f"expected {self.feature_dimension}, got {features.shape[1]}")
            
            # 尝试调整维度
            if features.shape[1] > self.feature_dimension:
                features = features[:, :self.feature_dimension]
            else:
                padding = np.zeros((features.shape[0], self.feature_dimension - features.shape[1]))
                features = np.hstack([features, padding])
            
            print(f"Adjusted feature dimension to: {features.shape}")
        
        # 3. 应用标准化（如果存在）
        if self.scaler is not None:
            print("Applying feature scaling...")
            features = self.scaler.transform(features)
        else:
            print("No feature scaling applied")
        
        return features
    
    def predict_bindenergy(self, smiles_list):
        """
        预测BindEnergy值
        """
        print(f"Predicting BindEnergy for {len(smiles_list)} molecules...")
        
        # 1. 提取特征
        features, valid_indices, processed_smiles = self.extract_features_from_smiles(smiles_list)
        
        if features.shape[0] == 0:
            print("No valid features extracted")
            return [], [], []
        
        # 2. 处理特征
        processed_features = self.process_features(features)
        
        # 3. 检查特征维度
        if self.feature_dimension is not None and processed_features.shape[1] != self.feature_dimension:
            print(f"Error: Processed features dimension ({processed_features.shape[1]}) "
                  f"doesn't match model dimension ({self.feature_dimension})")
            return [], valid_indices, processed_smiles
        
        # 4. 进行预测
        predictions = self.loaded_model.predict(processed_features)
        
        print(f"Predictions completed for {len(predictions)} molecules")
        
        return predictions, valid_indices, processed_smiles
    
    def predict_from_file(self, input_file, output_path=None, smiles_column=None):
        """
        从文件读取SMILES并进行预测
        """
        print(f"Loading data from: {input_file}")
        
        # 读取数据
        if input_file.endswith('.csv'):
            df = pd.read_csv(input_file)
        else:
            df = pd.read_excel(input_file)
        
        print(f"Dataset shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        
        # 自动检测SMILES列
        if smiles_column is None:
            columns = df.columns.tolist()
            smiles_col = next((col for col in columns if 'smile' in col.lower()), None)
            if smiles_col is None:
                smiles_col = next((col for col in columns if 'SMILES' in col), None)
            if smiles_col is None and len(columns) > 0:
                smiles_col = columns[0]
        else:
            smiles_col = smiles_column
        
        if smiles_col not in df.columns:
            raise ValueError(f"SMILES column '{smiles_col}' not found in dataset")
        
        print(f"Using SMILES column: {smiles_col}")
        
        # 提取SMILES列表
        smiles_list = df[smiles_col].astype(str).tolist()
        
        # 获取所有列名用于结果保存
        original_columns = df.columns.tolist()
        
        # 进行预测
        predictions, valid_indices, processed_smiles = self.predict_bindenergy(smiles_list)
        
        if len(predictions) == 0:
            print("No predictions were made")
            return pd.DataFrame()
        
        # 创建结果DataFrame
        results_data = []
        for i, idx in enumerate(valid_indices):
            result_row = {}
            
            # 添加原始数据
            for col in original_columns:
                if idx < len(df):
                    result_row[col] = df.iloc[idx][col]
                else:
                    result_row[col] = None
            
            # 添加预测结果
            result_row['BindEnergy_Predicted'] = predictions[i]
            result_row['Prediction_Index'] = i
            
            results_data.append(result_row)
        
        predictions_df = pd.DataFrame(results_data)
        
        # 重新排列列，使预测结果在最后
        col_order = [col for col in original_columns if col in predictions_df.columns] + ['Prediction_Index', 'BindEnergy_Predicted']
        predictions_df = predictions_df[col_order]
        
        # 保存结果
        if output_path:
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            predictions_df.to_csv(output_path, index=False, encoding='utf-8-sig')
            print(f"Predictions saved to: {output_path}")
        
        return predictions_df


# 主程序
if __name__ == "__main__":
    # 设置路径
    model_path = r"E:\a\dou\SAM\vs3\bindenergy"  # 模型目录
    input_file = r"E:\a\dou\SAM\vs3\verify\verify.csv"  # 输入文件
    output_file = r"E:\a\dou\SAM\vs3\predictions\bindenergy_predictions.csv"  # 输出文件
    
    print("=" * 80)
    print("BindEnergy Prediction System")
    print("=" * 80)
    
    # 检查文件是否存在
    if not os.path.exists(model_path):
        print(f"Error: Model path does not exist: {model_path}")
        exit(1)
    
    if not os.path.exists(input_file):
        print(f"Error: Input file does not exist: {input_file}")
        exit(1)
    
    # 创建输出目录
    output_dir = os.path.dirname(output_file)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # 创建预测器并进行预测
    try:
        # 初始化预测器
        predictor = BindEnergy_Predictor(model_path)
        
        # 从文件进行预测
        print(f"\nProcessing input file: {input_file}")
        results = predictor.predict_from_file(
            input_file=input_file,
            output_path=output_file,
            smiles_column=None
        )
        
        if len(results) > 0:
            # 显示统计信息
            print("\n" + "=" * 80)
            print("Prediction Results Summary:")
            print("=" * 80)
            
            # 读取输入文件获取总行数
            if input_file.endswith('.csv'):
                input_df = pd.read_csv(input_file)
            else:
                input_df = pd.read_excel(input_file)
            
            print(f"Total molecules in input: {len(input_df)}")
            print(f"Successfully predicted: {len(results)}")
            
            if 'BindEnergy_Predicted' in results.columns:
                pred_vals = results['BindEnergy_Predicted']
                print(f"\nPrediction Statistics:")
                print(f"Mean predicted BindEnergy: {pred_vals.mean():.4f}")
                print(f"Std predicted BindEnergy: {pred_vals.std():.4f}")
                print(f"Min predicted BindEnergy: {pred_vals.min():.4f}")
                print(f"Max predicted BindEnergy: {pred_vals.max():.4f}")
                
                # 显示所有预测结果
                print(f"\nAll predictions:")
                for i, row in results.iterrows():
                    mol_name = row.get('name', f"Molecule_{i+1}")
                    smiles_short = str(row.get('smiles', 'N/A'))[:50]
                    if len(str(row.get('smiles', ''))) > 50:
                        smiles_short += "..."
                    
                    print(f"  {i+1}. {mol_name}")
                    print(f"     SMILES: {smiles_short}")
                    print(f"     Predicted BindEnergy: {row['BindEnergy_Predicted']:.4f}")
                    print()
        
        print(f"\nDetailed results saved to: {output_file}")
        
    except Exception as e:
        print(f"\nError during prediction: {str(e)}")
        import traceback
        traceback.print_exc()
        exit(1)
    
    print("\n" + "=" * 80)
    print("Prediction completed!")
    print("=" * 80)

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
from sklearn.preprocessing import StandardScaler

# Try to import RDKit
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, MACCSkeys
    from rdkit.ML.Descriptors import MoleculeDescriptors
    from rdkit.Chem import Fragments
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False
    print("Warning: RDKit not installed, will use simplified feature extraction methods")

warnings.filterwarnings('ignore')

class ChargePredictor:
    def __init__(self, model_path, output_path):
        self.model_path = model_path
        self.output_path = output_path
        os.makedirs(output_path, exist_ok=True)
        
        # 加载已训练好的模型
        self.load_trained_model()
    
    def load_trained_model(self):
        """加载已训练好的模型"""
        print(f"Loading trained model from: {self.model_path}")
        
        try:
            # 加载模型文件
            model_info = joblib.load(self.model_path)
            
            # 提取模型和相关信息
            self.model = model_info['model']
            self.scaler = model_info.get('scaler', None)
            self.best_model_name = model_info.get('best_model_name', 'Unknown')
            self.feature_dimension = model_info.get('feature_dimension', 0)
            self.selected_feature_indices = model_info.get('selected_feature_indices', None)
            self.dim_reduction_method = model_info.get('dim_reduction_method', None)
            
            print(f"Model loaded successfully: {self.best_model_name}")
            print(f"Feature dimension: {self.feature_dimension}")
            print(f"Dimensionality reduction method: {self.dim_reduction_method}")
            
            if self.selected_feature_indices is not None:
                print(f"Number of selected features: {len(self.selected_feature_indices)}")
            
        except Exception as e:
            print(f"Error loading model: {e}")
            raise
    
    def extended_feature_engineering(self, smiles_list):
        """扩展特征工程 - 与训练代码相同的特征提取方法"""
        print("Performing extended feature engineering...")
        
        features = []
        valid_smiles = []
        
        for i, smiles in enumerate(smiles_list):
            try:
                if not isinstance(smiles, str):
                    smiles = str(smiles)
                
                mol = Chem.MolFromSmiles(smiles)
                if mol is not None:
                    # 1. Extended molecular fingerprints (与训练代码相同)
                    # Morgan fingerprints (different radii)
                    fp_1 = AllChem.GetMorganFingerprintAsBitVect(mol, 1, nBits=256)
                    fp_2 = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=256)
                    fp_3 = AllChem.GetMorganFingerprintAsBitVect(mol, 3, nBits=128)
                    
                    # RDKit topological fingerprint
                    rdkit_fp = Chem.RDKFingerprint(mol, fpSize=256)
                    
                    # MACCS fingerprint
                    maccs_fp = MACCSkeys.GenMACCSKeys(mol)
                    
                    # Atom pair fingerprint
                    ap_fp = AllChem.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=256)
                    
                    # Topological torsion fingerprint
                    tt_fp = AllChem.GetHashedTopologicalTorsionFingerprintAsBitVect(mol, nBits=256)
                    
                    # 2. Extended molecular descriptors
                    descriptor_names = [x[0] for x in Descriptors._descList]
                    descriptor_calc = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)
                    descriptors = list(descriptor_calc.CalcDescriptors(mol))
                    
                    # 3. Advanced descriptor calculation (跳过需要构象的描述符)
                    advanced_descriptors = []
                    
                    try:
                        # Charge-related descriptors
                        advanced_descriptors.append(rdMolDescriptors.CalcHallKierAlpha(mol))
                        advanced_descriptors.append(rdMolDescriptors.CalcKappa1(mol))
                        advanced_descriptors.append(rdMolDescriptors.CalcKappa2(mol))
                        advanced_descriptors.append(rdMolDescriptors.CalcKappa3(mol))
                        
                        # Aromaticity descriptors
                        advanced_descriptors.append(rdMolDescriptors.CalcNumAromaticRings(mol))
                        advanced_descriptors.append(rdMolDescriptors.CalcNumAliphaticRings(mol))
                        advanced_descriptors.append(rdMolDescriptors.CalcNumHeterocycles(mol))
                        
                        # Synthetic accessibility
                        advanced_descriptors.append(rdMolDescriptors.CalcNumBridgeheadAtoms(mol))
                        advanced_descriptors.append(rdMolDescriptors.CalcNumSpiroAtoms(mol))
                        
                    except Exception as e:
                        # 如果高级描述符计算失败，填充零
                        advanced_descriptors.extend([0] * 10)  # 10个高级描述符
                    
                    # 4. Molecular graph features
                    graph_features = []
                    try:
                        # Calculate molecular graph
                        adj_matrix = Chem.GetAdjacencyMatrix(mol)
                        degree = np.sum(adj_matrix, axis=1)
                        
                        graph_features.append(np.mean(degree))
                        graph_features.append(np.std(degree))
                        graph_features.append(np.max(degree))
                        graph_features.append(np.min(degree))
                        
                        # Ring features
                        ring_info = mol.GetRingInfo()
                        graph_features.append(ring_info.NumRings())
                        
                        # Atom type statistics
                        atoms = [atom.GetAtomicNum() for atom in mol.GetAtoms()]
                        graph_features.append(len(set(atoms)))  # Number of different atom types
                        
                    except Exception as e:
                        graph_features.extend([0] * 6)
                    
                    # 5. Molecular fragment features - 修正属性名
                    fragment_features = []
                    try:
                        # Common functional group counts - 使用正确的属性名
                        fragment_features.append(Fragments.fr_Al_COO(mol))
                        fragment_features.append(Fragments.fr_Al_OH(mol))
                        fragment_features.append(Fragments.fr_ArN(mol))
                        fragment_features.append(Fragments.fr_COO(mol))
                        fragment_features.append(Fragments.fr_C_O(mol))
                        fragment_features.append(Fragments.fr_NH2(mol))
                        fragment_features.append(Fragments.fr_OH(mol))
                        fragment_features.append(Fragments.fr_SH(mol))
                        fragment_features.append(Fragments.fr_aldehyde(mol))
                        fragment_features.append(Fragments.fr_ketone(mol))
                        fragment_features.append(Fragments.fr_ester(mol))
                        fragment_features.append(Fragments.fr_ether(mol))
                        fragment_features.append(Fragments.fr_amide(mol))
                    except Exception as e:
                        # 如果片段特征计算失败，尝试替代方法
                        try:
                            fragment_features = [
                                rdMolDescriptors.CalcNumAliphaticCarboxylicAcids(mol),
                                rdMolDescriptors.CalcNumAliphaticAlcohols(mol),
                                0,  # fr_ArN 没有直接替代
                                rdMolDescriptors.CalcNumCarboxylicAcids(mol),
                                rdMolDescriptors.CalcNumCarbonyls(mol),
                                rdMolDescriptors.CalcNumAmines(mol),
                                rdMolDescriptors.CalcNumAlcohols(mol),
                                rdMolDescriptors.CalcNumSulfur(mol),
                                rdMolDescriptors.CalcNumAldehydes(mol),
                                rdMolDescriptors.CalcNumKetones(mol),
                                rdMolDescriptors.CalcNumEsters(mol),
                                rdMolDescriptors.CalcNumEthers(mol),
                                rdMolDescriptors.CalcNumAmides(mol)
                            ]
                        except:
                            fragment_features.extend([0] * 13)
                    
                    # 6. Combine all features (与训练代码相同)
                    combined_features = np.concatenate([
                        np.array(fp_1),
                        np.array(fp_2),
                        np.array(fp_3),
                        np.array(rdkit_fp),
                        np.array(maccs_fp),
                        np.array(ap_fp),
                        np.array(tt_fp),
                        np.array(descriptors),
                        np.array(advanced_descriptors),
                        np.array(graph_features),
                        np.array(fragment_features)
                    ])
                    
                    # Handle possible NaN values
                    combined_features = np.nan_to_num(combined_features, nan=0.0, posinf=0.0, neginf=0.0)
                    
                    # 确保特征维度为1837（与训练时相同）
                    if len(combined_features) != 1837:
                        # 调整特征维度
                        if len(combined_features) > 1837:
                            combined_features = combined_features[:1837]
                        else:
                            # 填充零到1837维
                            padding = np.zeros(1837 - len(combined_features))
                            combined_features = np.concatenate([combined_features, padding])
                    
                    features.append(combined_features)
                    valid_smiles.append(smiles)
                    
                else:
                    print(f"Warning: SMILES {smiles} could not be parsed by RDKit")
                    # 添加一个零向量（1837维）
                    features.append(np.zeros(1837))
                    valid_smiles.append(smiles)
                    
            except Exception as e:
                print(f"Error processing SMILES {smiles}: {str(e)}")
                # 为无效的SMILES添加一个零向量（1837维）
                features.append(np.zeros(1837))
                valid_smiles.append(smiles)
        
        features_array = np.array(features)
        print(f"Feature engineering completed: {features_array.shape}")
        
        return features_array, valid_smiles
    
    def apply_same_feature_processing(self, X):
        """应用与训练时相同的特征处理流程"""
        print(f"\nApplying same feature processing as training...")
        print(f"Input feature dimensions: {X.shape}")
        
        # 根据训练时使用的处理方式进行处理
        if self.dim_reduction_method:
            print(f"Using dimensionality reduction method: {self.dim_reduction_method}")
            # 这里简化处理，实际应该保存并加载降维模型
            from sklearn.decomposition import PCA
            pca = PCA(n_components=self.feature_dimension)
            X_processed = pca.fit_transform(X)
        
        elif self.selected_feature_indices is not None:
            print(f"Using feature selection with {len(self.selected_feature_indices)} features")
            
            # 确保索引在有效范围内
            valid_indices = []
            for idx in self.selected_feature_indices:
                if idx < X.shape[1]:
                    valid_indices.append(idx)
                else:
                    print(f"Warning: Feature index {idx} out of range (max: {X.shape[1]-1})")
            
            if len(valid_indices) > 0:
                X_processed = X[:, valid_indices]
            else:
                print("Warning: No valid feature indices, using first 30 features")
                X_processed = X[:, :self.feature_dimension]
        
        else:
            # 如果没有特殊处理，使用前30个特征
            print(f"No specific processing method found, using first {self.feature_dimension} features")
            X_processed = X[:, :self.feature_dimension]
        
        print(f"Processed feature dimensions: {X_processed.shape}")
        
        return X_processed
    
    def prepare_features(self, smiles_list):
        """准备特征 - 与训练代码相同的流程"""
        print("\nPreparing features for prediction...")
        
        if not RDKIT_AVAILABLE:
            raise ImportError("RDKit not available, cannot perform feature engineering")
        
        # 提取扩展特征
        X, valid_smiles = self.extended_feature_engineering(smiles_list)
        print(f"Initial feature dimensions: {X.shape}")
        
        # 应用与训练时相同的特征处理
        X_processed = self.apply_same_feature_processing(X)
        
        print(f"Final feature dimensions: {X_processed.shape}")
        
        return X_processed, valid_smiles
    
    def predict(self, X):
        """使用加载的模型进行预测"""
        print(f"\nMaking predictions using {self.best_model_name}...")
        
        # 检查特征维度是否匹配
        if X.shape[1] != self.feature_dimension:
            print(f"Warning: Feature dimension mismatch! Input: {X.shape[1]}, Expected: {self.feature_dimension}")
            print("Attempting to adjust feature dimensions...")
            
            if X.shape[1] > self.feature_dimension:
                # 如果特征太多，截断
                X = X[:, :self.feature_dimension]
                print(f"Truncated to {X.shape[1]} features")
            else:
                # 如果特征太少，填充零
                padding = np.zeros((X.shape[0], self.feature_dimension - X.shape[1]))
                X = np.hstack([X, padding])
                print(f"Padded to {X.shape[1]} features")
        
        # 应用标准化（如果模型训练时使用了标准化）
        if self.scaler is not None:
            print("Applying standardization...")
            X_processed = self.scaler.transform(X)
        else:
            X_processed = X
        
        # 进行预测
        predictions = self.model.predict(X_processed)
        
        print(f"Predictions completed for {len(predictions)} molecules")
        
        return predictions
    
    def load_and_predict(self, data_path):
        """加载数据并进行预测"""
        print("=" * 80)
        print("Charge Transfer Prediction for New Molecules")
        print("=" * 80)
        
        # 1. 加载数据
        print(f"\n[1/3] Loading data from: {data_path}")
        
        try:
            if data_path.endswith('.xlsx') or data_path.endswith('.xls'):
                df = pd.read_excel(data_path)
            else:
                df = pd.read_csv(data_path)
        except Exception as e:
            print(f"Data loading failed: {e}")
            return None
        
        print(f"Dataset shape: {df.shape}")
        print(f"Column names: {list(df.columns)}")
        
        # 自动检测SMILES列名
        columns = df.columns.tolist()
        smiles_col = None
        
        # 尝试找到SMILES列
        possible_names = ['smiles', 'SMILES', 'smile', 'SMILES', 'smi', 'SMI']
        for col in columns:
            if col.lower() in [name.lower() for name in possible_names]:
                smiles_col = col
                break
        
        if smiles_col is None and len(columns) > 0:
            smiles_col = columns[0]
            print(f"Warning: Could not find SMILES column, using first column: {smiles_col}")
        
        if smiles_col is None:
            print("Error: No SMILES column found in data")
            return None
        
        print(f"Using SMILES column: {smiles_col}")
        smiles_list = df[smiles_col].astype(str).tolist()
        
        # 2. 准备特征
        print(f"\n[2/3] Preparing features for {len(smiles_list)} molecules...")
        try:
            X, valid_smiles = self.prepare_features(smiles_list)
        except Exception as e:
            print(f"Feature preparation failed: {e}")
            import traceback
            traceback.print_exc()
            return None
        
        # 3. 进行预测
        print(f"\n[3/3] Making predictions...")
        predictions = self.predict(X)
        
        # 4. 保存结果
        print(f"\nSaving prediction results...")
        
        # 创建结果DataFrame
        results_df = pd.DataFrame({
            'SMILES': valid_smiles,
            'Predicted_Charge': predictions
        })
        
        # 添加其他列（如果原始数据中有）
        for col in df.columns:
            if col != smiles_col and col not in results_df.columns:
                results_df[col] = df[col].values
        
        # 保存到文件
        output_file = os.path.join(self.output_path, 'charge_predictions.csv')
        results_df.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"Predictions saved to: {output_file}")
        
        # 显示统计信息
        print(f"\nPrediction Statistics:")
        print(f"  Number of molecules: {len(predictions)}")
        print(f"  Mean predicted charge: {predictions.mean():.4f}")
        print(f"  Std predicted charge: {predictions.std():.4f}")
        print(f"  Min predicted charge: {predictions.min():.4f}")
        print(f"  Max predicted charge: {predictions.max():.4f}")
        
        # 显示预测结果
        print(f"\nPrediction results:")
        for i in range(min(10, len(predictions))):
            if len(valid_smiles[i]) > 30:
                smiles_display = valid_smiles[i][:30] + "..."
            else:
                smiles_display = valid_smiles[i]
            print(f"  {i+1:2d}. SMILES: {smiles_display:<35} Predicted: {predictions[i]:.4f}")
        
        print("\n" + "=" * 80)
        print("Prediction complete!")
        print("=" * 80)
        
        return results_df


# 主程序
if __name__ == "__main__":
    # 设置路径
    model_path = r"E:\a\dou\SAM\vs3\charge\best_advanced_model.pkl"
    verify_path = r"E:\a\dou\SAM\vs3\verify\verify.csv"
    output_path = r"E:\a\dou\SAM\vs3\predictions"
    
    print(f"Model file: {model_path}")
    print(f"Verification data: {verify_path}")
    print(f"Output path: {output_path}")
    
    # 检查文件是否存在
    if not os.path.exists(model_path):
        print(f"Error: Model file does not exist: {model_path}")
        print("Please check if the file path is correct")
    elif not os.path.exists(verify_path):
        print(f"Error: Verification data does not exist: {verify_path}")
        print("Please check if the file path is correct")
    else:
        # 创建预测器并进行预测
        try:
            predictor = ChargePredictor(model_path, output_path)
            results = predictor.load_and_predict(verify_path)
            
            if results is not None:
                print(f"\nPrediction completed successfully!")
                print(f"Results saved to: {output_path}")
                
        except Exception as e:
            print(f"\nPrediction error: {str(e)}")
            import traceback
            traceback.print_exc()

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')

# Try to import RDKit
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, MACCSkeys
    from rdkit.ML.Descriptors import MoleculeDescriptors
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False
    print("Error: RDKit not installed. Please install RDKit to run predictions.")

class UnifiedPredictor:
    def __init__(self, model_path, property_type='charge'):
        """
        统一的预测器，可以预测所有四个属性
        
        Args:
            model_path: 模型文件的完整路径
            property_type: 预测的属性类型 ('charge', 'vbm', 'cbm', 'bindenergy')
        """
        self.model_path = model_path
        self.property_type = property_type
        self.model = None
        self.scaler = None
        self.feature_dimension = None
        self.selected_feature_indices = None
        self.dim_reduction_method = None
        self.pca_model = None
        
        # 加载模型
        self.load_model()
    
    def load_model(self):
        """加载保存的模型"""
        print(f"Loading model from: {self.model_path}")
        
        if not os.path.exists(self.model_path):
            raise FileNotFoundError(f"Model file not found: {self.model_path}")
        
        try:
            model_info = joblib.load(self.model_path)
            
            # 提取模型信息
            self.model = model_info.get('model')
            self.scaler = model_info.get('scaler')
            self.best_model_name = model_info.get('best_model_name', 'Unknown')
            self.feature_dimension = model_info.get('feature_dimension')
            self.selected_feature_indices = model_info.get('selected_feature_indices')
            self.dim_reduction_method = model_info.get('dim_reduction_method')
            
            # 尝试加载PCA模型
            self.pca_model = model_info.get('pca_model')
            
            # 显示加载的信息
            print(f"Model: {self.best_model_name}")
            print(f"Property type: {self.property_type}")
            print(f"Feature dimension: {self.feature_dimension}")
            
            if self.selected_feature_indices is not None:
                print(f"Selected feature indices: {len(self.selected_feature_indices)}")
            if self.dim_reduction_method:
                print(f"Dimensionality reduction: {self.dim_reduction_method}")
            if self.pca_model is not None:
                print(f"PCA model loaded: {self.pca_model.n_components_} components")
            
        except Exception as e:
            print(f"Error loading model: {e}")
            raise
    
    def extract_features(self, smiles_list):
        """
        从SMILES字符串提取特征
        使用与训练代码相同的特征提取方法
        """
        if not RDKIT_AVAILABLE:
            raise ImportError("RDKit not available")
        
        print(f"Extracting features for {len(smiles_list)} molecules...")
        
        features = []
        valid_smiles = []
        
        for i, smiles in enumerate(smiles_list):
            try:
                smiles = str(smiles)
                mol = Chem.MolFromSmiles(smiles)
                if mol is None:
                    print(f"Warning: Could not parse SMILES: {smiles[:50]}")
                    # 为无效分子创建零向量
                    features.append(np.zeros(1837))
                    valid_smiles.append(smiles)
                    continue
                
                # 1. Morgan指纹
                fp_1 = AllChem.GetMorganFingerprintAsBitVect(mol, 1, nBits=256)
                fp_2 = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=256)
                fp_3 = AllChem.GetMorganFingerprintAsBitVect(mol, 3, nBits=128)
                
                # 2. RDKit指纹
                rdkit_fp = Chem.RDKFingerprint(mol, fpSize=256)
                
                # 3. MACCS指纹
                maccs_fp = MACCSkeys.GenMACCSKeys(mol)
                
                # 4. 分子描述符
                descriptor_names = [x[0] for x in Descriptors._descList]
                descriptor_calc = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)
                descriptors = list(descriptor_calc.CalcDescriptors(mol))
                
                # 5. 组合所有特征
                combined_features = np.concatenate([
                    np.array(fp_1),
                    np.array(fp_2),
                    np.array(fp_3),
                    np.array(rdkit_fp),
                    np.array(maccs_fp),
                    np.array(descriptors)
                ])
                
                # 确保特征维度为1837
                if len(combined_features) > 1837:
                    combined_features = combined_features[:1837]
                elif len(combined_features) < 1837:
                    # 填充零
                    padding = np.zeros(1837 - len(combined_features))
                    combined_features = np.concatenate([combined_features, padding])
                
                # 处理NaN值
                combined_features = np.nan_to_num(combined_features, nan=0.0, posinf=0.0, neginf=0.0)
                
                features.append(combined_features)
                valid_smiles.append(smiles)
                
            except Exception as e:
                print(f"Error processing SMILES {smiles[:50]}: {e}")
                features.append(np.zeros(1837))
                valid_smiles.append(smiles)
        
        if not features:
            raise ValueError("No features extracted")
        
        features_array = np.array(features)
        print(f"Extracted features shape: {features_array.shape}")
        
        return features_array, valid_smiles
    
    def process_features(self, features):
        """
        处理特征：应用与训练时相同的处理流程
        """
        print(f"Processing features...")
        print(f"Input shape: {features.shape}")
        
        # 1. 特征选择
        if self.selected_feature_indices is not None:
            print(f"Applying feature selection with {len(self.selected_feature_indices)} indices")
            
            # 确保索引在范围内
            valid_indices = []
            for idx in self.selected_feature_indices:
                if idx < features.shape[1]:
                    valid_indices.append(idx)
                else:
                    print(f"Warning: Feature index {idx} out of range")
            
            if valid_indices:
                features = features[:, valid_indices]
            else:
                print("Warning: No valid feature indices, using all features")
        
        # 2. 降维处理
        if self.dim_reduction_method == 'PCA':
            print("Applying PCA dimensionality reduction")
            
            if self.pca_model is not None:
                # 使用保存的PCA模型
                features = self.pca_model.transform(features)
            else:
                # 如果没有保存的PCA模型，创建一个新的
                print("Warning: No saved PCA model, creating new one")
                pca = PCA(n_components=self.feature_dimension)
                features = pca.fit_transform(features)
            
            print(f"After PCA shape: {features.shape}")
        
        # 3. 检查特征维度
        if self.feature_dimension is not None and features.shape[1] != self.feature_dimension:
            print(f"Warning: Feature dimension mismatch. Expected: {self.feature_dimension}, Got: {features.shape[1]}")
            
            if features.shape[1] > self.feature_dimension:
                features = features[:, :self.feature_dimension]
            else:
                padding = np.zeros((features.shape[0], self.feature_dimension - features.shape[1]))
                features = np.hstack([features, padding])
            
            print(f"Adjusted shape: {features.shape}")
        
        # 4. 标准化
        if self.scaler is not None:
            print("Applying standardization")
            features = self.scaler.transform(features)
        
        return features
    
    def predict(self, smiles_list):
        """
        进行预测
        """
        print(f"Predicting {self.property_type} for {len(smiles_list)} molecules...")
        
        # 1. 提取特征
        features, valid_smiles = self.extract_features(smiles_list)
        
        # 2. 处理特征
        processed_features = self.process_features(features)
        
        # 3. 检查特征维度
        if self.feature_dimension is not None and processed_features.shape[1] != self.feature_dimension:
            print(f"Error: Final feature dimension mismatch. Expected: {self.feature_dimension}, Got: {processed_features.shape[1]}")
            return [], []
        
        # 4. 进行预测
        predictions = self.model.predict(processed_features)
        
        print(f"Predictions completed")
        
        return predictions, valid_smiles
    
    def predict_from_file(self, input_file, output_path=None):
        """
        从文件读取SMILES并进行预测
        """
        print(f"Loading data from: {input_file}")
        
        # 读取数据
        if input_file.endswith('.csv'):
            df = pd.read_csv(input_file)
        elif input_file.endswith(('.xlsx', '.xls')):
            df = pd.read_excel(input_file)
        else:
            raise ValueError("Unsupported file format")
        
        print(f"Dataset shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        
        # 检测SMILES列
        columns = df.columns.tolist()
        smiles_col = None
        
        for col in columns:
            if 'smile' in col.lower():
                smiles_col = col
                break
        
        if smiles_col is None and len(columns) > 0:
            smiles_col = columns[0]
            print(f"Warning: Using first column as SMILES: {smiles_col}")
        
        if smiles_col not in df.columns:
            raise ValueError(f"SMILES column '{smiles_col}' not found")
        
        print(f"Using SMILES column: {smiles_col}")
        
        # 提取SMILES列表
        smiles_list = df[smiles_col].astype(str).tolist()
        
        # 进行预测
        predictions, valid_smiles = self.predict(smiles_list)
        
        if len(predictions) == 0:
            print("No predictions made")
            return pd.DataFrame()
        
        # 创建结果DataFrame
        results = pd.DataFrame({
            'SMILES': valid_smiles,
            f'{self.property_type.upper()}_Predicted': predictions
        })
        
        # 添加原始列
        for col in df.columns:
            if col != smiles_col and col not in results.columns:
                results[col] = df[col].values
        
        # 保存结果
        if output_path:
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            results.to_csv(output_path, index=False, encoding='utf-8-sig')
            print(f"Results saved to: {output_path}")
        
        return results

def predict_all_properties(verify_path, output_dir):
    """
    预测所有四个属性
    """
    # 模型路径
    model_paths = {
        'charge': r"E:\a\dou\SAM\vs3\charge\best_advanced_model.pkl",
        'vbm': r"E:\a\dou\SAM\vs3\vbm\best_advanced_model.pkl",
        'cbm': r"E:\a\dou\SAM\vs3\cbm\best_advanced_model.pkl",
        'bindenergy': r"E:\a\dou\SAM\vs3\bindenergy\best_advanced_model.pkl"
    }
    
    # 输出文件路径
    output_files = {
        'charge': os.path.join(output_dir, 'charge_predictions.csv'),
        'vbm': os.path.join(output_dir, 'vbm_predictions.csv'),
        'cbm': os.path.join(output_dir, 'cbm_predictions.csv'),
        'bindenergy': os.path.join(output_dir, 'bindenergy_predictions.csv')
    }
    
    # 检查模型文件是否存在
    for prop, path in model_paths.items():
        if not os.path.exists(path):
            print(f"Warning: Model file for {prop} not found: {path}")
    
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    
    results = {}
    
    for prop in ['charge', 'vbm', 'cbm', 'bindenergy']:
        print("\n" + "="*80)
        print(f"Predicting {prop.upper()}")
        print("="*80)
        
        model_path = model_paths[prop]
        
        if not os.path.exists(model_path):
            print(f"Skipping {prop}: model file not found")
            continue
        
        try:
            # 创建预测器
            predictor = UnifiedPredictor(model_path, prop)
            
            # 进行预测
            result = predictor.predict_from_file(
                input_file=verify_path,
                output_path=output_files[prop]
            )
            
            if len(result) > 0:
                results[prop] = result
                
                # 显示统计信息
                pred_col = f'{prop.upper()}_Predicted'
                if pred_col in result.columns:
                    pred_values = result[pred_col]
                    print(f"\n{prop.upper()} Prediction Statistics:")
                    print(f"  Count: {len(pred_values)}")
                    print(f"  Mean: {pred_values.mean():.4f}")
                    print(f"  Std: {pred_values.std():.4f}")
                    print(f"  Min: {pred_values.min():.4f}")
                    print(f"  Max: {pred_values.max():.4f}")
                    
                    # 显示每个分子的预测值
                    print(f"\nIndividual predictions:")
                    for i, row in result.iterrows():
                        smiles = row['SMILES'][:50] + "..." if len(row['SMILES']) > 50 else row['SMILES']
                        print(f"  {i+1}: SMILES: {smiles:<60} {prop.upper()}: {row[pred_col]:.4f}")
            
        except Exception as e:
            print(f"Error predicting {prop}: {e}")
            import traceback
            traceback.print_exc()
    
    # 合并所有结果
    if results:
        merged_result = None
        
        for prop, df in results.items():
            if merged_result is None:
                merged_result = df.copy()
            else:
                # 基于SMILES列合并
                merge_cols = [col for col in df.columns if col not in merged_result.columns or col == 'SMILES']
                merged_result = pd.merge(
                    merged_result, 
                    df[merge_cols], 
                    on='SMILES', 
                    how='outer'
                )
        
        if merged_result is not None:
            merged_file = os.path.join(output_dir, 'all_predictions.csv')
            merged_result.to_csv(merged_file, index=False, encoding='utf-8-sig')
            print(f"\nMerged results saved to: {merged_file}")
            
            # 显示合并结果
            print(f"\nMerged Prediction Results:")
            print(merged_result[[col for col in merged_result.columns if 'Predicted' in col or 'SMILES' in col]].head())
    
    return results

# 主程序
if __name__ == "__main__":
    # 设置路径
    verify_path = r"E:\a\dou\SAM\vs3\verify\verify.csv"
    output_dir = r"E:\a\dou\SAM\vs3\predictions"
    
    print("="*80)
    print("Unified Prediction System for All Properties")
    print("="*80)
    
    # 检查输入文件
    if not os.path.exists(verify_path):
        print(f"Error: Input file not found: {verify_path}")
        exit(1)
    
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    
    # 预测所有属性
    results = predict_all_properties(verify_path, output_dir)
    
    print("\n" + "="*80)
    print("Prediction completed!")
    print("="*80)